In [1]:
%pip install -U langgraph langchain-google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [5]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [11]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call the llm gen outline
    prompt = f"Generate a detailed outlline for a blog on the topc - {title}"
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline
    return state

In [12]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"
    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [13]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog') 
graph.add_edge('create_blog', END) 

workflow = graph.compile()


In [14]:
intial_state = {'title': 'rise of AI in India'}
final_state = workflow.invoke(intial_state)
print(final_state)

{'title': 'rise of AI in India', 'outline': 'Okay, here\'s a detailed outline for a blog post on the "Rise of AI in India," designed to be engaging, informative, and well-structured.\n\n---\n\n## Blog Title Options:\n\n*   **India\'s AI Ascent: Unpacking the Digital Revolution Reshaping the Nation** (Primary Choice)\n*   From Silicon Valley to Bengaluru: How India is Redefining the Global AI Landscape\n*   The AI Renaissance: India\'s Journey Towards Becoming a Global AI Powerhouse\n*   Beyond the Hype: The Real Story of AI\'s Explosive Growth in India\n\n---\n\n## Detailed Blog Outline: India\'s AI Ascent\n\n**I. Introduction: The Dawn of a New Era**\n\n*   **A. Hook:** Start with a compelling statistic or a vivid image of AI\'s global impact, then pivot to India\'s unique position.\n    *   *Example:* "While the world grapples with the transformative power of Artificial Intelligence, a quiet revolution is unfolding in India, poised to reshape its economy, society, and global standing